In [1]:
import os
from pyspark.sql import SparkSession
from pyspark import SparkConf
from pyspark import SparkContext
import pyspark.sql.functions as F

# Variables d'environnement
os.environ['PYSPARK_SUBMIT_ARGS'] = (
    '--packages org.apache.hadoop:hadoop-aws:3.2.0,org.apache.hadoop:hadoop-common:3.2.0,io.trino:trino-jdbc:422 '
    'pyspark-shell'
)
os.environ['S3_ENDPOINT'] = "http://minio:9000"
os.environ['AWS_ACCESS_KEY_ID'] = "minio"
os.environ['AWS_SECRET_ACCESS_KEY'] = "minio123"

# On reprend la config S3 de l'autre fichier
spark = (
    SparkSession.builder
    .appName("spark-silver-to-gold")
    .config("spark.hadoop.fs.s3a.access.key", os.getenv("AWS_ACCESS_KEY_ID"))
    .config("spark.hadoop.fs.s3a.secret.key", os.getenv("AWS_SECRET_ACCESS_KEY"))
    .config("spark.hadoop.fs.s3a.endpoint", os.getenv("S3_ENDPOINT"))
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.attempts.maximum", "1")
    .config("spark.hadoop.fs.s3a.connection.establish.timeout", "5000")
    .config("spark.hadoop.fs.s3a.connection.timeout", "10000")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

#Comme c'est bien foutu on peut juste lire la table en parquet comme on lisait le json
silver_path = "s3a://velib/silver/velib-disponibilite-en-temps-reel"
df_silver = spark.read.parquet(silver_path)

# Verifications
print("Schema:")
df_silver.printSchema()
print("\nSample data:")
df_silver.show(5, truncate=False)
print("\nTotal records:", df_silver.count())

:: loading settings :: url = jar:file:/usr/local/spark-3.1.2-bin-hadoop3.2/jars/ivy-2.4.0.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jovyan/.ivy2/cache
The jars for the packages stored in: /home/jovyan/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
org.apache.hadoop#hadoop-common added as a dependency
io.trino#trino-jdbc added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-811cdc57-5fe9-4af0-8d3e-18a10a8f6fa7;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.2.0 in central
	found com.amazonaws#aws-java-sdk-bundle;1.11.375 in central
	found org.apache.hadoop#hadoop-common;3.2.0 in central
	found org.apache.hadoop#hadoop-annotations;3.2.0 in central
	found com.google.guava#guava;11.0.2 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found commons-cli#commons-cli;1.2 in central
	found org.apache.commons#commons-math3;3.1.1 in central
	found org.apache.httpcomponents#httpclient;4.5.2 in central
	found org.apache.httpcomponents#httpcore;4.4.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	f

Schema:
root
 |-- capacity: long (nullable = true)
 |-- duedate: string (nullable = true)
 |-- ebike: long (nullable = true)
 |-- is_installed: string (nullable = true)
 |-- is_renting: string (nullable = true)
 |-- is_returning: string (nullable = true)
 |-- mechanical: long (nullable = true)
 |-- name: string (nullable = true)
 |-- nom_arrondissement_communes: string (nullable = true)
 |-- numbikesavailable: long (nullable = true)
 |-- numdocksavailable: long (nullable = true)
 |-- stationcode: string (nullable = true)
 |-- fill_ratio: double (nullable = true)
 |-- part_minute: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- part_day: date (nullable = true)


Sample data:


+--------+-------------------------+-----+------------+----------+------------+----------+-----------------------------+---------------------------+-----------------+-----------------+-----------+-------------------+----------------+-----------------+------------------+----------+
|capacity|duedate                  |ebike|is_installed|is_renting|is_returning|mechanical|name                         |nom_arrondissement_communes|numbikesavailable|numdocksavailable|stationcode|fill_ratio         |part_minute     |lat              |lon               |part_day  |
+--------+-------------------------+-----+------------+----------+------------+----------+-----------------------------+---------------------------+-----------------+-----------------+-----------+-------------------+----------------+-----------------+------------------+----------+
|20      |2024-10-29T14:10:38+00:00|4    |OUI         |OUI       |OUI         |4         |Rouget de L'isle - Watteau   |Vitry-sur-Seine            |8     


Total records: 1274874


In [2]:
# Filter df_silver for part_day on or after 2024-10-28
df_silver = df_silver.filter(F.col("part_day") >= "2024-10-29")
df_silver = df_silver.withColumn("time", F.date_format("part_minute", "HH:mm"))

df_silver.show()

+--------+--------------------+-----+------------+----------+------------+----------+--------------------+---------------------------+-----------------+-----------------+-----------+-------------------+----------------+------------------+------------------+----------+-----+
|capacity|             duedate|ebike|is_installed|is_renting|is_returning|mechanical|                name|nom_arrondissement_communes|numbikesavailable|numdocksavailable|stationcode|         fill_ratio|     part_minute|               lat|               lon|  part_day| time|
+--------+--------------------+-----+------------+----------+------------+----------+--------------------+---------------------------+-----------------+-----------------+-----------+-------------------+----------------+------------------+------------------+----------+-----+
|      20|2024-10-29T14:10:...|    4|         OUI|       OUI|         OUI|         4|Rouget de L'isle ...|            Vitry-sur-Seine|                8|               12|     

## Turnover rate par station par range de temps

In [3]:
from pyspark.sql.window import Window
 
# Define the window partitioned by station and ordered by time

window_spec = Window.partitionBy("name").orderBy("duedate")
 
# Calculate the turnover rate (absolute change in bike availability)

df_turnover = df_silver.withColumn("prev_numbikesavailable", F.lag("numbikesavailable").over(window_spec)) \
.withColumn("turnover_rate", F.abs(F.col("numbikesavailable") - F.col("prev_numbikesavailable"))) \
.groupBy("name", "part_day", "time") \
.agg(F.sum("turnover_rate").alias("total_turnover"))
 
df_turnover.show()
 

+--------------------+----------+-----+--------------+
|                name|  part_day| time|total_turnover|
+--------------------+----------+-----+--------------+
|Camille Groult - ...|2024-10-29|09:07|             0|
|Camille Groult - ...|2024-10-29|10:07|            10|
|Camille Groult - ...|2024-10-29|12:08|             2|
|Camille Groult - ...|2024-10-29|13:09|            20|
|Camille Groult - ...|2024-10-29|14:09|            25|
|Camille Groult - ...|2024-10-29|15:10|            10|
|Camille Groult - ...|2024-10-29|23:06|            11|
|Camille Groult - ...|2024-10-30|07:10|             5|
|Camille Groult - ...|2024-10-30|08:12|             1|
|Camille Groult - ...|2024-10-30|09:12|            41|
|Camille Groult - ...|2024-10-30|10:11|            21|
|Camille Groult - ...|2024-10-30|11:12|             1|
|Camille Groult - ...|2024-10-30|12:12|             1|
|Camille Groult - ...|2024-10-30|13:13|             1|
|Camille Groult - ...|2024-10-30|14:13|            14|
|Camille G

## Stats de base sur l'ensemble des stations par range de temps

In [4]:
# Aggregate statistics per station
df_stats_all = df_silver.groupBy("part_day", "time") \
    .agg(
        F.avg("numbikesavailable").alias("avg_bikes"),
        F.expr("percentile(numbikesavailable, 0.5)").alias("median_bikes"),
        F.min("numbikesavailable").alias("min_bikes"),
        F.max("numbikesavailable").alias("max_bikes"),
        F.avg("fill_ratio").alias("avg_fill_ratio")
    ) \
    .orderBy("part_day", "time")

In [5]:
df_stats_all.show()

+----------+-----+------------------+------------+---------+---------+-------------------+
|  part_day| time|         avg_bikes|median_bikes|min_bikes|max_bikes|     avg_fill_ratio|
+----------+-----+------------------+------------+---------+---------+-------------------+
|2024-10-29|09:01|6.7696969696969695|         4.0|        0|       23|0.25476164340164953|
|2024-10-29|09:02| 4.478813559322034|         2.0|        0|       20|0.17572779153486348|
|2024-10-29|09:03|3.2305825242718447|         2.0|        0|       20|0.13211865555378616|
|2024-10-29|09:04| 5.423510466988728|         2.0|        0|       52|0.17989517521353335|
|2024-10-29|09:05|  6.75645342312009|         4.0|        0|       34| 0.2429572943560061|
|2024-10-29|09:06|  6.95141065830721|         4.0|        0|       30|0.25852051024807415|
|2024-10-29|09:07| 8.572062084257206|         5.0|        0|       54|0.29338877573162075|
|2024-10-29|09:08|10.894536817102138|         7.0|        0|       44| 0.3591062730059128|

## Stats de base par station par range de temps

In [6]:
# Aggregate statistics per station
df_stats_station = df_silver.groupBy("name", "part_day", "time", "lat", "lon") \
    .agg(
        F.avg("numbikesavailable").alias("avg_bikes"),
        F.expr("percentile(numbikesavailable, 0.5)").alias("median_bikes"),
        F.min("numbikesavailable").alias("min_bikes"),
        F.max("numbikesavailable").alias("max_bikes"),
        F.avg("fill_ratio").alias("avg_fill_ratio")
    )

In [7]:
df_stats_station.show()

+--------------------+----------+-----+------------------+------------------+------------------+------------+---------+---------+-------------------+
|                name|  part_day| time|               lat|               lon|         avg_bikes|median_bikes|min_bikes|max_bikes|     avg_fill_ratio|
+--------------------+----------+-----+------------------+------------------+------------------+------------+---------+---------+-------------------+
|                null|2024-10-30|14:13|              null|              null|15.031862745098039|        11.0|        0|       65|               null|
|                null|2024-10-31|09:46|              null|              null| 4.081081081081081|         2.0|        0|       26|               null|
|Alexander Fleming...|2024-10-31|08:47| 48.88179064200294|2.4030599377642674|2.1923076923076925|         2.0|        2|        3|0.07071960297766748|
|André Maurois - J...|2024-10-30|12:14|        48.8778983|         2.2789591|              10.0|    

In [8]:
df_stats_station.filter(F.col("name") == "Argenteuil - Voltaire").show()

+--------------------+----------+-----+-----------------+------------------+------------------+------------+---------+---------+-------------------+
|                name|  part_day| time|              lat|               lon|         avg_bikes|median_bikes|min_bikes|max_bikes|     avg_fill_ratio|
+--------------------+----------+-----+-----------------+------------------+------------------+------------+---------+---------+-------------------+
|Argenteuil - Volt...|2024-10-31|10:13|48.91864865955558|2.2814112156629562| 6.445783132530121|         6.0|        6|        7|0.40286144578313254|
|Argenteuil - Volt...|2024-10-30|14:14|48.91864865955558|2.2814112156629562|              10.9|        11.0|       10|       11|            0.68125|
|Argenteuil - Volt...|2024-10-30|15:14|48.91864865955558|2.2814112156629562|              11.6|        12.0|       11|       12|              0.725|
|Argenteuil - Volt...|2024-10-29|14:05|48.91864865955558|2.2814112156629562| 3.672727272727273|         4.

## Update hive metastore

In [9]:
df_stats_station.printSchema()

root
 |-- name: string (nullable = true)
 |-- part_day: date (nullable = true)
 |-- time: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lon: double (nullable = true)
 |-- avg_bikes: double (nullable = true)
 |-- median_bikes: double (nullable = true)
 |-- min_bikes: long (nullable = true)
 |-- max_bikes: long (nullable = true)
 |-- avg_fill_ratio: double (nullable = true)



In [10]:
import trino

host, port, user = 'trino-coordinator', 8080, 'trino'
conn = trino.dbapi.connect(host=host, port=port, user=user)
cur = conn.cursor()

def createTable(df, catalog, schema_name, table, schema_location, schema, partitioned_by, external_location):
    os.environ['S3_OUTPUT_PATH'] = external_location

    spark.sparkContext.setLogLevel("WARN")

    # Write DataFrame to S3
    (
        df.write
        .partitionBy(partitioned_by)
        .format("parquet")
        .mode("overwrite")
        .save(os.getenv('S3_OUTPUT_PATH'))
    )
    
    
    host, port, user = 'trino-coordinator', 8080, 'trino'
    conn = trino.dbapi.connect(host=host, port=port, user=user)
    cur = conn.cursor()
    
    queries = [
        f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema_name} WITH (location = '{schema_location}')",
        f"DROP TABLE IF EXISTS {catalog}.{schema_name}.{table}",
        f"""
        CREATE TABLE IF NOT EXISTS {catalog}.{schema_name}.{table} (
        {schema}
        )
        WITH (
            format = 'PARQUET',
            partitioned_by = ARRAY['{partitioned_by}'],
            external_location = '{external_location}'
        )
        """,
        f"USE {catalog}.{schema_name}",
        f"CALL system.sync_partition_metadata('{schema_name}', '{table}', 'ADD')",
        # f"CALL system.sync_partition_metadata('{catalog}.{schema_name}.{table}', 'ADD')",
        f"SELECT * FROM {catalog}.{schema_name}.{table} LIMIT 5"
    ]
    
    # Execute each query in the list
    for query in queries:
        try:
            cur.execute(query)
            # Check if the query is a SELECT query to fetch results
            if query.startswith("SELECT"):
                results = cur.fetchall()
                for row in results:
                    print(row)
            else:
                print(f"Executed: {query}")
        except Exception as e:
            print(f"Error executing query: {query}. Error: {e}")

    # Close the cursor and connection
    cur.close()
    conn.close()
    
schema_turnover = f"""
    part_day DATE,
    time VARCHAR,
    total_turnover BIGINT,
    name VARCHAR
"""

schema_stats_all = f"""
    time VARCHAR,
    avg_bikes DOUBLE,
    median_bikes DOUBLE,
    min_bikes BIGINT,
    max_bikes BIGINT,
    avg_fill_ratio DOUBLE,
    part_day DATE
"""

schema_stats_station = f"""
    name VARCHAR,
    time VARCHAR,
    lat DOUBLE,
    lon DOUBLE,
    avg_bikes DOUBLE,
    median_bikes DOUBLE,
    min_bikes BIGINT,
    max_bikes BIGINT,
    avg_fill_ratio DOUBLE,
    part_day DATE
"""

createTable(df_turnover, 'minio', 'velib_gold_turnover', 'velib_disponibilite_en_temps_reel_turnover', 's3a://velib/gold/', schema_turnover, 'name', 's3a://velib/gold/velib-disponibilite-en-temps-reel-turnover')
createTable(df_stats_all, 'minio', 'velib_gold_stats_all', 'velib_disponibilite_en_temps_reel_stats_all', 's3a://velib/gold/', schema_stats_all, 'part_day', 's3a://velib/gold/velib-disponibilite-en-temps-reel-stats_all')
# createTable(df_stats_station, 'minio', 'velib_gold_stats_station', 'velib_disponibilite_en_temps_reel_stats_station', 's3a://velib/gold/', schema_stats_station, 'name', 's3a://velib/gold/velib-disponibilite-en-temps-reel-stats_station')

24/10/31 15:27:24 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
24/10/31 15:27:24 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
24/10/31 15:27:24 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
24/10/31 15:27:24 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
24/10/31 15:27:24 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
24/10/31 15:27:24 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
24/10/31 15:27:24 WARN MemoryManager: Total allocation exceeds 95.

Executed: CREATE SCHEMA IF NOT EXISTS minio.velib_gold_turnover WITH (location = 's3a://velib/gold/')
Executed: DROP TABLE IF EXISTS minio.velib_gold_turnover.velib_disponibilite_en_temps_reel_turnover
Executed: 
        CREATE TABLE IF NOT EXISTS minio.velib_gold_turnover.velib_disponibilite_en_temps_reel_turnover (
        
    part_day DATE,
    time VARCHAR,
    total_turnover BIGINT,
    name VARCHAR

        )
        WITH (
            format = 'PARQUET',
            partitioned_by = ARRAY['name'],
            external_location = 's3a://velib/gold/velib-disponibilite-en-temps-reel-turnover'
        )
        
Executed: USE minio.velib_gold_turnover
Executed: CALL system.sync_partition_metadata('velib_gold_turnover', 'velib_disponibilite_en_temps_reel_turnover', 'ADD')
[datetime.date(2024, 10, 29), '09:11', 4, 'Aubervilliers - Curial']
[datetime.date(2024, 10, 29), '09:06', 5, 'Bercy - Traversiére']
[datetime.date(2024, 10, 29), '09:10', 8, 'Beaumarchais - Pas de la Mule']
[datet

24/10/31 15:29:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
24/10/31 15:29:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
24/10/31 15:29:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
24/10/31 15:29:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
24/10/31 15:29:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
24/10/31 15:29:07 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
24/10/31 15:29:07 WARN MemoryManager: Total allocation exceeds 95.

Executed: CREATE SCHEMA IF NOT EXISTS minio.velib_gold_stats_all WITH (location = 's3a://velib/gold/')
Executed: DROP TABLE IF EXISTS minio.velib_gold_stats_all.velib_disponibilite_en_temps_reel_stats_all
Executed: 
        CREATE TABLE IF NOT EXISTS minio.velib_gold_stats_all.velib_disponibilite_en_temps_reel_stats_all (
        
    time VARCHAR,
    avg_bikes DOUBLE,
    median_bikes DOUBLE,
    min_bikes BIGINT,
    max_bikes BIGINT,
    avg_fill_ratio DOUBLE,
    part_day DATE

        )
        WITH (
            format = 'PARQUET',
            partitioned_by = ARRAY['part_day'],
            external_location = 's3a://velib/gold/velib-disponibilite-en-temps-reel-stats_all'
        )
        
Executed: USE minio.velib_gold_stats_all
Executed: CALL system.sync_partition_metadata('velib_gold_stats_all', 'velib_disponibilite_en_temps_reel_stats_all', 'ADD')
['14:08', 9.165008099976857, 5.0, 0, 58, 0.3008500150238236, datetime.date(2024, 10, 29)]
['09:12', 13.232632665254467, 10.0, 0,

In [11]:
createTable(df_stats_station, 'minio', 'velib_gold_stats_station', 'velib_disponibilite_en_temps_reel_stats_station', 's3a://velib/gold/', schema_stats_station, 'part_day', 's3a://velib/gold/velib-disponibilite-en-temps-reel-stats_station')

24/10/31 15:29:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
24/10/31 15:29:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
24/10/31 15:29:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
24/10/31 15:29:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
24/10/31 15:29:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
24/10/31 15:29:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
24/10/31 15:29:33 WARN MemoryManager: Total allocation exceeds 95.

Executed: CREATE SCHEMA IF NOT EXISTS minio.velib_gold_stats_station WITH (location = 's3a://velib/gold/')
Executed: DROP TABLE IF EXISTS minio.velib_gold_stats_station.velib_disponibilite_en_temps_reel_stats_station
Executed: 
        CREATE TABLE IF NOT EXISTS minio.velib_gold_stats_station.velib_disponibilite_en_temps_reel_stats_station (
        
    name VARCHAR,
    time VARCHAR,
    lat DOUBLE,
    lon DOUBLE,
    avg_bikes DOUBLE,
    median_bikes DOUBLE,
    min_bikes BIGINT,
    max_bikes BIGINT,
    avg_fill_ratio DOUBLE,
    part_day DATE

        )
        WITH (
            format = 'PARQUET',
            partitioned_by = ARRAY['part_day'],
            external_location = 's3a://velib/gold/velib-disponibilite-en-temps-reel-stats_station'
        )
        
Executed: USE minio.velib_gold_stats_station
Executed: CALL system.sync_partition_metadata('velib_gold_stats_station', 'velib_disponibilite_en_temps_reel_stats_station', 'ADD')
['André Mazet - Saint-André des Arts', '10

In [12]:
df_stats_station.count()

32058